In [2]:
from stackapi import StackAPI
import pandas as pd
import time  # To manage rate limits

# Initialize the StackAPI
SITE = StackAPI('stackoverflow')
SITE.max_pages = 1  # Restrict to one page (adjustable)
SITE.page_size = 100  # Maximum allowed page size

# Fetch questions with specific tags
def fetch_questions(tag, min_words=None, sort='votes'):
    """
    Fetch questions tagged with the specified tag.
    Optionally filter by minimum number of words in the title.
    """
    questions = SITE.fetch('questions', tagged=tag, sort=sort, order='desc')
    data = []
    
    for item in questions['items']:
        if min_words is None or len(item['title'].split()) <= min_words:
            accepted_answer_id = item.get('accepted_answer_id', None)
            accepted_answer_body = None
            if accepted_answer_id:
                try:
                    # Fetch the accepted answer with the body included
                    answer = SITE.fetch(
                        f'answers/{accepted_answer_id}', 
                        filter='withbody'
                    )
                    accepted_answer_body = answer['items'][0].get('body', None) if 'items' in answer and answer['items'] else None
                except Exception as e:
                    print(f"Error fetching accepted answer for question {item['question_id']}: {e}")
            
            data.append({
                "Title": item['title'],
                "Link": item['link'],
                "Views": item['view_count'],
                "Votes": item['score'],
                "Creation Date": item['creation_date'],
                "Accepted Answer Body": accepted_answer_body,
            })
            time.sleep(0.1)  # Pause to respect API rate limits
    
    return data

# Fetch datasets
java_active_high_votes = fetch_questions('java', sort='votes')
python_active_high_votes = fetch_questions('python', sort='votes')
java_short_high_votes = fetch_questions('java', min_words=5, sort='votes')
python_short_high_votes = fetch_questions('python', min_words=5, sort='votes')

# Save datasets into CSV files
datasets = {
    "Java_Active_High_Votes": java_active_high_votes,
    "Python_Active_High_Votes": python_active_high_votes,
    "Java_Short_High_Votes": java_short_high_votes,
    "Python_Short_High_Votes": python_short_high_votes,
}

for name, data in datasets.items():
    df = pd.DataFrame(data)
    df.to_csv(f"{name}.csv", index=False)
    print(f"Saved {name}.csv with {len(data)} records.")


Saved Java_Active_High_Votes.csv with 100 records.
Saved Python_Active_High_Votes.csv with 100 records.
Saved Java_Short_High_Votes.csv with 11 records.
Saved Python_Short_High_Votes.csv with 9 records.
